In [1]:
import os
import sys
sys.path.append(os.path.abspath('..'))
import logging
import warnings
import re
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from src.build_dataset import get_file_pairs, merge_qa_data, detect_exercise_type, find_answer_index, apply_reference_tag

# --- Setup Warnings ---
warnings.filterwarnings('ignore', category=SyntaxWarning, message='invalid escape sequence')

# --- Setup Logger ---
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)

# --- Load Environment Variables ---
load_dotenv()

c:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [2]:
data_path = os.getenv("DATA_DIR")

if data_path:
    data_dir = Path(data_path)
    logger.info(f"Ξεκινάει η αναζήτηση στον φάκελο: {data_dir}")
    
    all_pairs = get_file_pairs (data_dir, target_school="GEL")
    logger.info(f"Βρέθηκαν συνολικά {len(all_pairs)} ζευγάρια αρχείων (JSON/MD).")
else:
    logger.error("Το DATA_DIR δεν βρέθηκε στο .env αρχείο!")

2026-04-06 12:54:49 - INFO - Ξεκινάει η αναζήτηση στον φάκελο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\data
2026-04-06 12:54:49 - INFO - Βρέθηκαν συνολικά 59 ζευγάρια αρχείων (JSON/MD).


In [3]:
main_dataset = []

for pair in all_pairs:
    json_path = pair["json"]
    md_path = pair["md"]
    
    qa_list = merge_qa_data(json_path, md_path)
    
    main_dataset.extend(qa_list)

logger.info (f"Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά {len(main_dataset)} ερωτήσεις-απαντήσεις!")

2026-04-06 12:54:54 - INFO - Η ενοποίηση ολοκληρώθηκε! Βρέθηκαν συνολικά 1318 ερωτήσεις-απαντήσεις!


In [4]:
subject_translation = {
    "nea_ellinika": "greek_language",
    "arxaia": "ancient_greek",
    "istoria": "history",
    "latinika": "latin",
    "biologia": "biology",
    "fysiki": "physics",
    "ximeia": "chemistry",
    "pliroforiki": "computer_science",
    "arxes_oikonomikis_theorias": "economics",
    "mathimatika": "mathematics"
}

In [5]:
for item in main_dataset:
    q_text = item.get("question","")
    q_choices = item.get("choices",[])
    ans_text = item.get("answer","")
    images_list = item.get("images", [])
    marks = item.get("mark", [])
    
    form_type = detect_exercise_type(q_text,q_choices)
    item["format"] = form_type
    ans_idx = find_answer_index(q_choices,ans_text)
    item["answer_index"] = ans_idx
    item["reference"] = apply_reference_tag(item)
    
    old_subj = item.get("subject", "")
    new_subj = subject_translation.get(old_subj, old_subj)
    item["subject"] = new_subj
    
    #parsing image description and transcription
    all_descriptions = []
    all_transcriptions = []
    all_paths = []
    
    for img_dict in images_list:
        desc = img_dict.get("description","")
        if desc:
            all_descriptions.append(desc)
        transc = img_dict.get("transcription",[])
        if transc and isinstance(transc, list):
            joined_transc = ", ".join(transc)
            all_transcriptions.append(joined_transc)
        
        img_path = img_dict.get("path", "")
        if img_path:
            all_paths.append(img_path)
    
    mark_list = []
    
    for mark_text in marks:
        match = re.search(r'\d+\.?\d*', str(mark_text))
        if match:
            num_str = match.group()
            if "." in num_str:
                mark_list.append(float(num_str))
            else:
                mark_list.append(int(num_str))
    
    if len(mark_list) == 1:
        item["points"] = mark_list[0]
    elif len(mark_list) > 1:
        item["points"] = sum(mark_list)
    else:
        item["points"] = None
            
    item["image_description"] = " | ".join(all_descriptions)
    item["image_transcription"] = " | ".join(all_transcriptions)
    item["images"] = all_paths
    item.pop("mark", None)
    
    year = item.get("year", "")
    old_id = item.get("id", "")
    school_type = str(item.get("school_type", "gel")).lower()
    item["id"] = f"{new_subj}_{school_type}_{year}_{old_id}"

In [6]:
images_found = 0
print("--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---")

for item in main_dataset:
    imgs = item.get("images", [])
    
    if isinstance(imgs, list) and len(imgs) > 0:
        images_found += 1
        print(f"ID: {item.get('id')} στο μάθημα {item.get('subject')} ({item.get('year')}) - Περιέχει {len(imgs)} εικόνα/ες")

print(f"\nΣυνολικά βρέθηκαν {images_found} ερωτήσεις (IDs) με εικόνες.")

--- Ερωτήσεις που βρέθηκαν να έχουν εικόνες ---
ID: economics_gel_2025_Γ1 στο μάθημα economics (2025) - Περιέχει 1 εικόνα/ες
ID: economics_gel_2025_Γ2 στο μάθημα economics (2025) - Περιέχει 1 εικόνα/ες
ID: economics_gel_2025_Γ3 στο μάθημα economics (2025) - Περιέχει 1 εικόνα/ες
ID: economics_gel_2025_Γ4 στο μάθημα economics (2025) - Περιέχει 1 εικόνα/ες
ID: biology_gel_2025_Γ1.α στο μάθημα biology (2025) - Περιέχει 1 εικόνα/ες
ID: biology_gel_2025_Γ1.β στο μάθημα biology (2025) - Περιέχει 1 εικόνα/ες
ID: biology_gel_2025_Γ1.γ στο μάθημα biology (2025) - Περιέχει 1 εικόνα/ες
ID: biology_gel_2025_Γ2 στο μάθημα biology (2025) - Περιέχει 1 εικόνα/ες
ID: biology_gel_2025_Δ1 στο μάθημα biology (2025) - Περιέχει 1 εικόνα/ες
ID: biology_gel_2025_Δ2 στο μάθημα biology (2025) - Περιέχει 2 εικόνα/ες
ID: biology_gel_2025_Δ3 στο μάθημα biology (2025) - Περιέχει 2 εικόνα/ες
ID: biology_gel_2025_Δ4 στο μάθημα biology (2025) - Περιέχει 2 εικόνα/ες
ID: biology_gel_2025_Δ5 στο μάθημα biology (2025) - Πε

In [7]:
results_dir = Path("../results")
results_dir.mkdir(parents=True, exist_ok=True)

output_file = results_dir / "panellinies_dataset.xlsx"

In [8]:
df = pd.DataFrame(main_dataset)
df = df.rename (columns={"answer": "answer_text"})
my_columns = [
    "id",
    "subject",
    "format",
    "reference",
    "question",
    "input",
    "images",
    "choices",
    "answer_text",
    "answer_index",
    "image_description",
    "image_transcription",
    "points",
    "year",
    "school_type"
]

df = df[my_columns]

In [9]:
df.to_excel(output_file, index=False)

logger.info (f"Tο αρχείο δημιουργήθηκε επιτυχώς στο: {output_file.resolve()}!")

df.head()

2026-04-06 12:55:22 - INFO - Tο αρχείο δημιουργήθηκε επιτυχώς στο: C:\Users\panag\Desktop\ΙΕΛ ΕΡΓΑΣΙΑ\Εξετάσεις_γλωσσομάθειας\Πανελλήνιες\panellinies_exams_dataset\results\panellinies_dataset.xlsx!


,id,subject,format,reference,question,input,images,choices,answer_text,answer_index,image_description,image_transcription,points,year,school_type
0,ancient_greek_gel_2025_Α1.α.1,ancient_greek,multiple_choice,passage,Ποια είναι η κύρια αιτία η οποία εμποδίζει του...,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],"[α. Οι αλυσίδες στους αυχένες τους., β. Το σκο...",α,0.0,,,2.0,2025,GEL
1,ancient_greek_gel_2025_Α1.α.2,ancient_greek,multiple_choice,passage,Πού βρίσκεται το πυρ σε σχέση με τους δεσμώτες;,Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],"[α. Μπροστά τους, χαμηλά., β. Επάνω και πίσω τ...",β,1.0,,,2.0,2025,GEL
2,ancient_greek_gel_2025_Α1.α.3,ancient_greek,multiple_choice,passage,"Τα «σκεύη», οι «ἀνδριάντες» και τα «ἄλλα ζῷα» ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],"[α. είναι πραγματικά ζώα της σπηλιάς., β. δημι...",β,1.0,,,2.0,2025,GEL
3,ancient_greek_gel_2025_Α1.β,ancient_greek,open_ended,passage,"«παρ’ ἣν», «ὑπὲρ ὧν»: Σε ποια λέξη του αρχαίου...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],παρ ́ἥν: αναφέρεται στην ὁδόν\nὑπέρ ὧν: αναφέρ...,NaN,,,4.0,2025,GEL
4,ancient_greek_gel_2025_B1,ancient_greek,open_ended,passage,"Ποιος είναι ο βασικός εκφραστικός τρόπος, με τ...",Πλάτωνος Πολιτεία (514a-515c)\n\nΜετὰ ταῦτα δή...,[],[],Ο κυριότερος εκφραστικός τρόπος με τον οποίο ο...,NaN,,,10.0,2025,GEL
